In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    import pandas as pd
import seaborn as sns
sns.set()

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
### cell 0 ###

df = pd.read_csv("/home/dias-benchmarks/notebooks/saisandeepjallepalli/adidas-retail-eda-data-visualization/input/adidas-us-retail-products-dataset/adidas.csv")
factor = 1000
df = pd.concat([df]*factor)
df.info()

In [ ]:
### cell 1 ###

# First 5 records in the DataFrame

df.head()

In [ ]:
### cell 2 ###

# Checking for Null Values in the DataFrame

df.isna().sum()

# There 16 Null Values for Original_price Column

In [ ]:
### cell 3 ###

# Shape of the DataFrame

df.shape

In [ ]:
### cell 4 ###

# Information about the DataFrame

df.info()

In [ ]:
### cell 5 ###

# Correlation Between Columns

num_df = df.select_dtypes(include=["number"])
numeric_corr = num_df.corr()
numeric_corr

In [ ]:
### cell 6 ###

# Statistical informtion about the DataFrame

df.describe().T

In [ ]:
### cell 7 ###

# The Percentage of Missing Values in the 'original_price' Column

100 * (df['original_price'].isna().sum() / len(df))

# As the percntage of Null Records is less than 5%, hence dropping the Null record Rows

In [ ]:
### cell 8 ###

# Dropping Null Values in the DataFrame

df.dropna(inplace=True, axis=0)

In [ ]:
### cell 9 ###

# Checking for Null Values in the DataFrame

df.isna().sum()

In [ ]:
### cell 10 ###

# Dropping 'currency' column as all records have 'USD' as currency
# Dropping 'source' column as all records have 'adidas United States' as value
# Dropping 'brand', 'country', 'language' columns as all records have same value

df.drop([ 'brand', 'country', 'language', 'source_website', 'images', 'crawled_at', 'url', 'sku', 'currency','source', 'description'], axis=1, inplace=True)

In [ ]:
### cell 11 ###

# First 5 records in the DataFrame

df.head()

In [ ]:
### cell 12 ###

# Removing '$' from the DataFrame in one GPU‐accelerated step
df['original_price'] = df['original_price'].str.lstrip('$')

In [ ]:
### cell 13 ###

# First 5 records in the DataFrame

df.head()

In [ ]:
### cell 14 ###

# Shape of the DataFrame

df.shape

In [ ]:
### cell 15 ###

# Checking the Data Types for the columns

df.dtypes

In [ ]:
### cell 16 ###

# Creating 'Category' Column
# Use GPU-friendly str.partition to extract everything before the first '/'
df['Category'] = df['breadcrumbs'].str.partition("/")[0]

In [ ]:
### cell 17 ###

# Creating 'Product Type' column in one GPU‐only pass
df['Product_Type'] = df['breadcrumbs'].str.split("/").list.get(1)

In [ ]:
### cell 18 ###

# Droping 'breadcrumbs' Column

df.drop(['breadcrumbs', 'category'], axis=1, inplace=True)

In [ ]:
### cell 19 ###

# Convert 'original_price' to int64 in a single astype call to reduce GPU round trips
df = df.astype({ 'original_price': 'int64' })

In [ ]:
### cell 20 ###

# Checking the Data Types for the columns

df.dtypes

In [ ]:
### cell 21 ###

# No.of Unique Products in the DataFrame

df['name'].nunique()

In [ ]:
### cell 22 ###

# Top 10 - Products sold in the DataFrame
df['name'].value_counts().nlargest(10)

In [ ]:
### cell 23 ###

# Top 5 - Products sold in the DataFrame (GPU-only operations)
top_products = df['name'].value_counts().head(5)
top_products

In [ ]:
### cell 24 ###

# No.of Unique selling price in the DataFrame

df['selling_price'].nunique()

In [ ]:
### cell 25 ###

# Top 15 products in terms of similar selling price using a GPU‐optimized head

df['selling_price'].value_counts().head(15)

In [ ]:
### cell 26 ###

# Top 5 - Products interms of similar selling price in the DataFrame

df['selling_price'].value_counts()[:5]

In [ ]:
### cell 27 ###

# Maximum Selling Price

print("The Maximun Selling Price is:",df['selling_price'].max(),"USD")

In [ ]:
### cell 28 ###

# Minimum Selling Price

print("The Minimum Selling Price is:",df['selling_price'].min(),"USD")

In [ ]:
### cell 29 ###

# Average Selling Price

print("The Average Selling Price is:",round(df['selling_price'].mean(),2),"USD")

In [ ]:
### cell 30 ###

# Top 10 - Highest Selling Prices in USD

set(df['selling_price'].sort_values(ascending=False)[:30].values)

In [ ]:
### cell 31 ###

# Top 10 - Least Selling Prices in USD

set(df['selling_price'].sort_values()[:40].values)

In [ ]:
### cell 32 ###

# No.of Unique original price in the DataFrame

df['original_price'].nunique()

In [ ]:
### cell 33 ###

# Top 15 - Products interms of similar original price in the DataFrame

df['original_price'].value_counts()[:15]

In [ ]:
### cell 34 ###

# Top 5 - Products interms of similar origial price in the DataFrame

df['original_price'].value_counts()[:5]

In [ ]:
### cell 35 ###

# Maximum Original Price

print("The Maximun Original Price is:",df['original_price'].max(),"USD")

In [ ]:
### cell 36 ###

# Minimum Original Price
print(f"The Minimum Original Price is: {df.original_price.min()} USD")

In [ ]:
### cell 37 ###

avg_price = df['original_price'].mean()
print(f"The Average Original Price is: {avg_price:.2f} USD")

In [ ]:
### cell 38 ###

# Top 10 - Highest Original Prices in USD

set(df['original_price'].sort_values(ascending=False)[:80].values)

In [ ]:
### cell 39 ###

# Top 100 - Least Original Prices in USD (GPU-optimized partial sort)
set(
    df['original_price']
      .nsmallest(100)    # GPU‐side partial sort
      .to_numpy()        # copy only 100 values to host as a NumPy array
)

In [ ]:
### cell 40 ###

# Calculating Discount

df['Discount'] = df['original_price'] - df['selling_price']

In [ ]:
### cell 41 ###

# No.of Unique Discount in the DataFrame

df['Discount'].nunique()

In [ ]:
### cell 42 ###

# Top 15 - Highest Discount Amount in the DataFrame

list(set(df['Discount'].unique()))[-15::]

In [ ]:
### cell 43 ###

# Top 15 - Least Discount Amount in the DataFrame

list(set(df['Discount'].unique()))[:15]

In [ ]:
### cell 44 ###

# Top 15 - Most Given Discount Amount in the DataFrame

df['Discount'].value_counts()[:15]

In [ ]:
### cell 45 ###

# Top 5 - Most Given Discount Amount in the DataFrame

df['Discount'].value_counts()[:5]

In [ ]:
### cell 46 ###

# Top 15 - Most Given Discount Amount in the DataFrame

labels = df['Discount'].value_counts().head(15).index
_ = df['Discount'].value_counts().head(15).values

In [ ]:
### cell 47 ###

# Calculating Discount percentage(%)

df['Discount(%)'] = round(((df['original_price'] - df['selling_price']) / (df['original_price']))*100,2)

In [ ]:
### cell 48 ###

# No.of Unique Discount Percentage(%) in the DataFrame

df['Discount(%)'].nunique()

In [ ]:
### cell 49 ###

# Top 15 - Highest Discount Percentage(%) in the DataFrame

top_discount_percnet = list(set(df['Discount(%)'].unique()))
top_discount_percnet.sort(reverse=True)
print(top_discount_percnet[:15])

In [ ]:
### cell 50 ###

# Top 15 - Least Discount Percentage(%) in the DataFrame

least_discount_percnet = list(set(df['Discount(%)'].unique()))
least_discount_percnet.sort(reverse=False)
print(least_discount_percnet[:15])

In [ ]:
### cell 51 ###

# Top 15 - Most Given Discount Percentage(%) in the DataFrame

df['Discount(%)'].value_counts().head(15)

In [ ]:
### cell 52 ###

# Top 5 - Most Given Discount Percentage in the DataFrame

df['Discount(%)'].value_counts().head(5)

In [ ]:
### cell 53 ###

# Top 15 - Most Given Discount Percentage in the DataFrame
counts = df['Discount(%)'].value_counts().head(15)
labels = counts.index
_= counts.values

In [ ]:
### cell 54 ###

# Unique colors in the DataFrame (GPU-accelerated)
df['color'].drop_duplicates()

In [ ]:
### cell 55 ###

# No.of Unique colors in the DataFrame

df['color'].nunique()

In [ ]:
### cell 56 ###

# No.of Products for each Color

df['color'].value_counts()

In [ ]:
### cell 57 ###

# No.of Products for each Color
# Compute value_counts once and reuse
ev_counts = df['color'].value_counts()
labels = ev_counts.index
_ = x = ev_counts.values

In [ ]:
### cell 58 ###

# Popular Color in Kids Category

df.groupby(['Category','color']).color.count().loc['Kids']

In [ ]:
### cell 59 ###

# Popular Color in Womens Category

df.groupby(['Category','color']).size()['Women']

In [ ]:
### cell 60 ###

# Popular Color in Mens Category

df.groupby(['Category','color']).size()['Men']

In [ ]:
### cell 61 ###

# Unique Availability values in the DataFrame (GPU)
# Use drop_duplicates which is implemented on GPU in cuDF

df['availability'].drop_duplicates()

In [ ]:
### cell 62 ###

# No.of Unique Availability values in the DataFrame

df['availability'].nunique()

In [ ]:
### cell 63 ###

# No.of Products according to Avaialability

df['availability'].value_counts()

In [ ]:
### cell 64 ###

# Unique Product Type values using GPU-accelerated drop_duplicates
unique_product_types = df['Product_Type'].drop_duplicates().reset_index(drop=True)
unique_product_types

In [ ]:
### cell 65 ###

# No.of Unique Product Type values in the DataFrame

df['Product_Type'].nunique()

In [ ]:
### cell 66 ###

# No.of Products according to Product Type

df['Product_Type'].value_counts()

In [ ]:
### cell 67 ###

# No.of Products according to Product Type

df['Product_Type'].value_counts()

In [ ]:
### cell 68 ###

# Compute value counts once to avoid redundant GPU operations
vc = df['Product_Type'].value_counts()
labels = vc.index
_ = vc.values

In [ ]:
### cell 69 ###

df.groupby(['Product_Type', 'Category'])['selling_price'].max()

In [ ]:
### cell 70 ###

df.groupby(['Product_Type', 'Category'])['selling_price'].min()

In [ ]:
### cell 71 ###

df.groupby(['Product_Type', 'Category']).mean(numeric_only=True)['selling_price']

In [ ]:
### cell 72 ###

# Unique Category values in the DataFrame using GPU
df['Category'].drop_duplicates()

In [ ]:
### cell 73 ###

# No.of Unique Category values in the DataFrame

df['Category'].nunique()

In [ ]:
### cell 74 ###

# No.of Products according to Category

df['Category'].value_counts()

In [ ]:
### cell 75 ###

# Top-5 Products according to Category

df['Category'].value_counts().head(5)

In [ ]:
### cell 76 ###

vc = df['Category'].value_counts().head(5)
labels = vc.index
_ = vc.values

In [ ]:
### cell 77 ###

# Unique 'Average Rating'in the DataFrame

df['average_rating'].unique()

In [ ]:
### cell 78 ###

# No.of Unique 'Average Rating'in the DataFrame

df['average_rating'].nunique()

In [ ]:
### cell 79 ###

# No.of time particular Average Rating Provided

df['average_rating'].value_counts()

In [ ]:
### cell 80 ###

# Top 5 - Average Rating Provided

df['average_rating'].value_counts().head(5)

In [ ]:
### cell 81 ###

# Compute value_counts once and reuse the result for top-5
_top5 = df['average_rating'].value_counts().head(5)
labels = _top5.index
_ = _top5.values

In [ ]:
### cell 82 ###

# Maxmimum Average Rating

df['average_rating'].max()

In [ ]:
### cell 83 ###

# Minimum Average Rating

df['average_rating'].min()

In [ ]:
### cell 84 ###

# Mean Average Rating

round(df['average_rating'].mean(),2)

In [ ]:
### cell 85 ###

df.groupby(['Product_Type', 'Category'])['average_rating'].max()

In [ ]:
### cell 86 ###

df.groupby(['Product_Type', 'Category'])['average_rating'].min()

In [ ]:
### cell 87 ###

# No.of Unique 'Reviews Count' in the DataFrame

df['reviews_count'].nunique()

In [ ]:
### cell 88 ###

# Top 10 - 'Reviews Count' and no.of particular 'Reviews Count' occurances

df['reviews_count'].value_counts().head(10)

In [ ]:
### cell 89 ###

# Top 5 - 'Reviews Count' and no.of particular 'Reviews Count' occurances

df['average_rating'].value_counts().head(5)

In [ ]:
### cell 90 ###

# Maxmimum 'Reviews Count'

df['reviews_count'].max()

In [ ]:
### cell 91 ###

# Minimum 'Reviews Count'

df['reviews_count'].min()

In [ ]:
### cell 92 ###

# Mean 'Reviews Count'

round(df['reviews_count'].mean(),2)

In [ ]:
### cell 93 ###

df.groupby(['Product_Type', 'Category'])['reviews_count'].max()

In [ ]:
### cell 94 ###

df.groupby(['Product_Type', 'Category'])['reviews_count'].min()

In [ ]:
### cell 95 ###

df.shape